In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.appName("Flight Data").getOrCreate()
sc = spark.sparkContext

In [3]:
import pyspark
print(pyspark.__version__)

3.5.0


In [ ]:
df = spark.read.csv("Flight Dataset.csv", header=True, inferSchema=True)

In [5]:
df.show(5)

+--------+---------+---------+--------+--------+---------+---------+
| FL_DATE|DEP_DELAY|ARR_DELAY|AIR_TIME|DISTANCE| DEP_TIME| ARR_TIME|
+--------+---------+---------+--------+--------+---------+---------+
|1/1/2006|        5|       19|     350|    2475| 9.083333|12.483334|
|1/2/2006|      167|      216|     343|    2475|11.783334|15.766666|
|1/3/2006|       -7|       -2|     344|    2475| 8.883333|12.133333|
|1/4/2006|       -5|      -13|     331|    2475| 8.916667|    11.95|
|1/5/2006|       -3|      -17|     321|    2475|     8.95|11.883333|
+--------+---------+---------+--------+--------+---------+---------+
only showing top 5 rows



In [6]:
df.printSchema()

root
 |-- FL_DATE: string (nullable = true)
 |-- DEP_DELAY: integer (nullable = true)
 |-- ARR_DELAY: integer (nullable = true)
 |-- AIR_TIME: integer (nullable = true)
 |-- DISTANCE: integer (nullable = true)
 |-- DEP_TIME: double (nullable = true)
 |-- ARR_TIME: double (nullable = true)



In [ ]:
#Function to standardize the column datatypes
from pyspark.sql.functions import col, to_date, floor, make_timestamp, year, month, dayofmonth

def fix_schema(df):
    # 1. Convert FL_DATE to DateType
    df = df.withColumn(
        "FL_DATE",
        to_date(col("FL_DATE"), "M/d/yyyy")
    )

    # 2. Extract hour and minute from decimal time
    df = df.withColumn("DEP_HOUR", floor(col("DEP_TIME"))) \
           .withColumn("DEP_MINUTE", ((col("DEP_TIME") - col("DEP_HOUR")) * 60).cast("int"))

    df = df.withColumn("ARR_HOUR", floor(col("ARR_TIME"))) \
           .withColumn("ARR_MINUTE", ((col("ARR_TIME") - col("ARR_HOUR")) * 60).cast("int"))

    # 3. Build timestamps using make_timestamp (CORRECT way)
    df = df.withColumn(
        "DEP_TIMESTAMP",
        make_timestamp(
            year("FL_DATE"),
            month("FL_DATE"),
            dayofmonth("FL_DATE"),
            col("DEP_HOUR"),
            col("DEP_MINUTE"),
            col("DEP_MINUTE") * 0  # seconds = 0
        )
    )

    df = df.withColumn(
        "ARR_TIMESTAMP",
        make_timestamp(
            year("FL_DATE"),
            month("FL_DATE"),
            dayofmonth("FL_DATE"),
            col("ARR_HOUR"),
            col("ARR_MINUTE"),
            col("ARR_MINUTE") * 0
        )
    )

    # 4. Drop helper columns
    df = df.drop("DEP_HOUR", "DEP_MINUTE", "ARR_HOUR", "ARR_MINUTE")

    return df

In [12]:
df_clean = fix_schema(df)

df_clean.printSchema()
df_clean.show(5, truncate=False)

root
 |-- FL_DATE: date (nullable = true)
 |-- DEP_DELAY: integer (nullable = true)
 |-- ARR_DELAY: integer (nullable = true)
 |-- AIR_TIME: integer (nullable = true)
 |-- DISTANCE: integer (nullable = true)
 |-- DEP_TIME: double (nullable = true)
 |-- ARR_TIME: double (nullable = true)
 |-- DEP_TIMESTAMP: timestamp (nullable = true)
 |-- ARR_TIMESTAMP: timestamp (nullable = true)

+----------+---------+---------+--------+--------+---------+---------+-------------------+-------------------+
|FL_DATE   |DEP_DELAY|ARR_DELAY|AIR_TIME|DISTANCE|DEP_TIME |ARR_TIME |DEP_TIMESTAMP      |ARR_TIMESTAMP      |
+----------+---------+---------+--------+--------+---------+---------+-------------------+-------------------+
|2006-01-01|5        |19       |350     |2475    |9.083333 |12.483334|2006-01-01 09:04:00|2006-01-01 12:29:00|
|2006-01-02|167      |216      |343     |2475    |11.783334|15.766666|2006-01-02 11:47:00|2006-01-02 15:45:00|
|2006-01-03|-7       |-2       |344     |2475    |8.883333 |

In [ ]:
#function that gives back how many flights arrived earlier than expected. 
from pyspark.sql.functions import col

def count_early_arrivals(df):
    return df.filter(col("ARR_DELAY") < 0).count()

In [14]:
early_arrivals = count_early_arrivals(df_clean)

In [20]:
display(early_arrivals/df_clean.count()*100)

53.4655

In [ ]:
#Function that determines the typical departure time for flights over 2000 miles. 
from pyspark.sql.functions import avg

def typical_departure_time_long_flights(df):
    return df.filter(col("DISTANCE") > 2000) \
             .agg(avg("DEP_TIME").alias("avg_dep_time")) \
             .collect()[0]["avg_dep_time"]

In [22]:
print(typical_departure_time_long_flights(df_clean))

13.973233947624635


In [ ]:
#Function that gives back the proportion of flights that have arrival delays longer than 60 minutes. 
def proportion_long_arrival_delays(df):
    total = df.count()
    delayed = df.filter(col("ARR_DELAY") > 60).count()
    return delayed / total if total > 0 else 0

In [34]:
print(proportion_long_arrival_delays(df_clean))

0.053066


In [ ]:
#Function that gives the average airtime for flights that left earlier than 9:00 am. 
def avg_airtime_before_9am(df):
    return df.filter(col("DEP_TIME") < 9.0) \
             .agg(avg("AIR_TIME").alias("avg_air_time")) \
             .collect()[0]["avg_air_time"]

In [28]:
print(avg_airtime_before_9am(df_clean))

111.36120276990287


In [ ]:
#Function that determines the maximum arrival delay for flights that did not experience a delay upon departure. 
from pyspark.sql.functions import max

def max_arrival_delay_no_dep_delay(df):
    return df.filter(col("DEP_DELAY") <= 0) \
             .agg(max("ARR_DELAY").alias("max_arr_delay")) \
             .collect()[0]["max_arr_delay"]

In [32]:
print(max_arrival_delay_no_dep_delay(df_clean))

701


Assumptions:

- Each row represents one flight (No duplicates unless explicitly present)
- Delay columns are in minutes
- DEP_DELAY and ARR_DELAY are measured in minutes (Negative values = early and Positive values = delayed)
- Time columns are in decimal hours (Example: 9.083333 ≈ 9:05 AM)
- Based on a 24-hour clock
- No timezone complications
- Departure and arrival times are assumed comparable (same reference frame)
- Null values are ignored
- Rows with nulls in relevant columns are either: filtered out, or assumed not present